# CyberBench — Qwen 2.5-3B RFT Training Pipeline

**Rejection-Sampling Fine-Tuning (RFT)** of Qwen2.5-3B-Instruct on cybersecurity incident response.

```
Scenario → Log Analyst → Vuln Scanner ──┐
                          Threat Intel  ──┤→ Orchestrator → Qwen2.5-3B
                                                                ↓
                                                       Judge (SBERT + Groq)
                                                                ↓
                                                           RFT Loop
```

| Component | Details |
|-----------|--------|
| Target Model | Qwen2.5-3B-Instruct (4-bit NF4) |
| Fine-tuning | LoRA r=16, alpha=32, Rejection-Sampling FT |
| Briefing agents | Log Analyst, Vuln Scanner, Threat Intel, Orchestrator (Groq) |
| Judge | Fine-tuned SBERT + Groq llama-3.3-70b |
| Scenarios | 10 cybersecurity incident cases |
| Runtime | ~45 min on T4 GPU (Colab free tier) |

> **Requirements:** Google Colab T4 GPU, HuggingFace token (optional, for push_to_hub)

In [ ]:
# Cell 1 — Install dependencies
# NOTE: After this cell finishes, go to Runtime → Restart session, then run from Cell 2.
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# torch is pre-installed in Colab — do NOT reinstall it (causes torchvision conflicts)
pip(
    "transformers>=4.44.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.32.0",
    "trl>=0.9.0",
    "datasets>=2.20.0",
    "sentence-transformers>=5.4.1",
    "huggingface_hub>=0.24.0",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "pandas",
)
print("✓ Dependencies installed — now go to Runtime → Restart session, then continue from Cell 2.")

In [ ]:
# Cell 2 — Clone repo
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/gitikraj/Meta-Hack-Final.git"
REPO_DIR = "/content/Meta-Hack-Final"

def _run(cmd, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result

if not Path(REPO_DIR).exists():
    print(f"Cloning {REPO_URL} ...")
    _run(f"git clone {REPO_URL} {REPO_DIR}")
else:
    print(f"Repo exists at {REPO_DIR} — pulling latest ...")
    try:
        _run(f"git -C {REPO_DIR} pull --ff-only")
    except RuntimeError:
        print("Pull failed — re-cloning ...")
        _run(f"rm -rf {REPO_DIR}")
        _run(f"git clone {REPO_URL} {REPO_DIR}")

if not Path(REPO_DIR).exists():
    raise RuntimeError(
        f"Repo not found at {REPO_DIR} after clone.\n"
        "Make sure the repo is PUBLIC on GitHub."
    )

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Cell 3 — Load secrets from Colab (GROQ_API_KEY required, HF_TOKEN optional)
import os

try:
    from google.colab import userdata
    groq_key = userdata.get("GROQ_API_KEY") or ""
    hf_token = userdata.get("HF_TOKEN") or ""
except Exception:
    groq_key = ""
    hf_token = ""

if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key
    print("GROQ_API_KEY : set")
else:
    print("GROQ_API_KEY : NOT set — add it to Colab secrets (key icon in left panel)")

os.environ["HF_TOKEN"] = hf_token
print(f"HF_TOKEN     : {'set' if hf_token else 'not set (push_to_hub disabled)'}")

In [ ]:
# Cell 4 — Download and fine-tune SBERT semantic judge
import os, json, math, random, shutil
from pathlib import Path

SBERT_MODEL_DIR = os.path.join(REPO_DIR, "sbert/model")

if Path(SBERT_MODEL_DIR).exists() and any(Path(SBERT_MODEL_DIR).iterdir()):
    print(f"SBERT model found at {SBERT_MODEL_DIR} — skipping training")
else:
    from sentence_transformers import SentenceTransformer, InputExample
    from sentence_transformers.losses import CosineSimilarityLoss
    from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
    from torch.utils.data import DataLoader

    print("Loading base SBERT model (all-MiniLM-L6-v2)...")
    model = SentenceTransformer("all-MiniLM-L6-v2")

    corpus_path = os.path.join(REPO_DIR, "sbert/corpus/cyber_pairs.json")
    with open(corpus_path, encoding="utf-8") as f:
        pairs = json.load(f)
    print(f"Corpus: {len(pairs)} pairs")

    random.seed(42)
    random.shuffle(pairs)
    split = max(1, int(len(pairs) * 0.9))
    train_pairs, eval_pairs = pairs[:split], pairs[split:]

    train_examples = [
        InputExample(texts=[p["sentence1"], p["sentence2"]], label=float(p["score"]))
        for p in train_pairs
    ]
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=16)
    loss_fn = CosineSimilarityLoss(model)

    evaluator = EmbeddingSimilarityEvaluator(
        [p["sentence1"] for p in eval_pairs],
        [p["sentence2"] for p in eval_pairs],
        [float(p["score"]) for p in eval_pairs],
        name="cyber-eval",
    )

    if Path(SBERT_MODEL_DIR).exists():
        shutil.rmtree(SBERT_MODEL_DIR)

    print("Fine-tuning SBERT on cybersecurity pairs (~2 min)...")
    model.fit(
        train_objectives=[(train_loader, loss_fn)],
        evaluator=evaluator,
        epochs=3,
        output_path=SBERT_MODEL_DIR,
        show_progress_bar=True,
    )
    print(f"SBERT saved to {SBERT_MODEL_DIR}")

In [ ]:
# Cell 5 — GPU / VRAM check
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if vram_gb < 12:
    print("Warning: < 12 GB VRAM — reduce episodes_per_round to 4")
else:
    print("Sufficient VRAM for Qwen2.5-3B with 4-bit quantization")


In [ ]:
# Cell 6 — Configure RFT training parameters
import os
from qwen_training.rft_trainer import RFTConfig

config = RFTConfig(
    model_name                  = "Qwen/Qwen2.5-3B-Instruct",
    load_in_4bit                = True,
    lora_rank                   = 16,        # LoRA rank
    lora_alpha                  = 32,        # LoRA alpha
    lora_dropout                = 0.05,
    rounds                      = 3,         # increase to 5-8 for better results
    episodes_per_round          = 6,         # scenarios sampled per round
    top_k_ratio                 = 0.5,       # keep top 50% for SFT
    min_score_threshold         = 40.0,      # discard episodes below this
    learning_rate               = 2e-4,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_train_epochs            = 2,         # SFT epochs per round
    max_seq_length              = 2048,
    output_dir                  = "qwen_training/checkpoints",
    push_to_hub                 = bool(os.environ.get("HF_TOKEN")),
    hub_repo                    = "your-username/cyberbench-qwen-3b",  # update this
)

print(f"RFT config: {config.rounds} rounds x {config.episodes_per_round} episodes")
print(f"LoRA: rank={config.lora_rank}, alpha={config.lora_alpha}")
print(f"Top-K ratio: {config.top_k_ratio} | Min score: {config.min_score_threshold}")
print(f"Push to Hub: {config.push_to_hub}")


In [ ]:
# Cell 7 — Load Qwen2.5-3B-Instruct baseline (no LoRA adapter yet)
import torch
from qwen_training.qwen_target_agent import QwenTargetAgent

print("Loading Qwen2.5-3B-Instruct with 4-bit NF4 quantization...")
print("First load downloads ~6 GB from HuggingFace (5-10 min)")

baseline_agent = QwenTargetAgent(
    model_name   = config.model_name,
    adapter_path = None,     # no adapter yet — this is the baseline
    load_in_4bit = True,
)
baseline_agent._load()  # force-load now

used  = torch.cuda.memory_allocated(0) / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Model loaded. VRAM: {used:.1f} / {total:.1f} GB used")


In [ ]:
# Cell 8 — Define local helpers (no Groq) + run baseline episode
import json, re
from pathlib import Path

# ── Build briefing directly from scenario data (no API calls) ──────────────
def build_briefing(case):
    logs = case.get("logs", {})
    env  = case.get("environment", {})
    assets = ", ".join(f"{a['hostname']} ({a['role']})" for a in env.get("assets", []))
    lines = [
        f"ENVIRONMENT: {env.get('os', 'Unknown')} | Network: {env.get('network_range', 'N/A')}",
        f"ASSETS: {assets}", "",
        "AUTH EVENTS:",
    ]
    for log in logs.get("auth_logs", [])[:15]:
        lines.append(f"  {log.get('ts','')} | {log.get('event','')} | user={log.get('user','')} | src={log.get('src_ip','')} | host={log.get('host','')}")
    if logs.get("system_logs"):
        lines.append("\nSYSTEM EVENTS:")
        for log in logs["system_logs"][:10]:
            lines.append(f"  {log.get('ts','')} | {log.get('event','')} | host={log.get('host','')} | {log.get('detail', log.get('process',''))}")
    if logs.get("network_logs"):
        lines.append("\nNETWORK EVENTS:")
        for log in logs["network_logs"][:10]:
            lines.append(f"  {log.get('ts','')} | {log.get('event','')} | {log.get('src_ip','')} → {log.get('dst_ip','')} | bytes={log.get('bytes_out', log.get('bytes','?'))}")
    return "\n".join(lines)

# ── SBERT-only judge (no Groq) ─────────────────────────────────────────────
def local_judge(case, response):
    from pipeline.semantic import get_scorer
    scorer = get_scorer()
    gt = case["known_truth"]

    # Technique match — keyword search
    truth_ids = {t["id"].upper() for t in gt.get("techniques", [])}
    found_ids = {tid for tid in truth_ids if tid in response.upper()}
    technique_score = (len(found_ids) / len(truth_ids) * 100) if truth_ids else 50.0

    # IOC match — exact string search
    truth_iocs = set(gt.get("iocs", {}).get("ips", [])) | set(gt.get("iocs", {}).get("domains", []))
    found_iocs = {ioc for ioc in truth_iocs if ioc in response}
    ioc_score = (len(found_iocs) / len(truth_iocs) * 100) if truth_iocs else 50.0

    # Action match — SBERT
    truth_actions = gt.get("correct_immediate_actions", [])
    resp_sentences = [s.strip() for s in re.split(r'\n|\.', response) if len(s.strip()) > 10]
    action_score = scorer.list_match(resp_sentences, truth_actions) * 100 if truth_actions else 50.0

    # Root cause — SBERT
    truth_rc = gt.get("root_cause", "")
    root_cause_score = scorer.similarity(response[:1000], truth_rc) * 100 if truth_rc else 50.0

    # Blast radius — SBERT
    truth_br = gt.get("blast_radius", "")
    blast_radius_score = scorer.similarity(response[:1000], truth_br) * 100 if truth_br else 50.0

    # Completeness — section heading heuristic
    section_kws = ["what happened", "current risk", "immediate action", "investigation", "remediation", "hardening"]
    resp_lower = response.lower()
    completeness_score = sum(1 for s in section_kws if s in resp_lower) / len(section_kws) * 100

    overall = round(
        technique_score * 0.25 + ioc_score * 0.20 + action_score * 0.20 +
        root_cause_score * 0.15 + blast_radius_score * 0.10 + completeness_score * 0.10, 1
    )
    verdict = "pass" if overall >= 75 else "partial" if overall >= 45 else "fail"
    return {
        "overall": overall, "verdict": verdict,
        "algorithmic": {
            "technique_match": round(technique_score, 1), "ioc_match": round(ioc_score, 1),
            "action_match": round(action_score, 1), "root_cause_match": round(root_cause_score, 1),
            "blast_radius_match": round(blast_radius_score, 1), "completeness": round(completeness_score, 1),
        }
    }

# ── Episode collector ──────────────────────────────────────────────────────
def collect_episode_local(case, agent):
    briefing = build_briefing(case)
    prompt   = agent.build_prompt({
        "agent_name": "Qwen-CyberBench",
        "agent_description": "A fine-tuned 3B cybersecurity incident response model.",
        "goal": case["goal"], "briefing": briefing,
    })
    response = agent.generate(prompt)
    result   = local_judge(case, response)
    return {
        "prompt": prompt, "response": response,
        "score": result["overall"], "verdict": result["verdict"],
        "case_id": case["case_id"], "difficulty": case["difficulty"],
        "category": case["category"], "goal": case["goal"],
        "judge_full": result,
    }

# ── Run baseline episode ───────────────────────────────────────────────────
with open("data/scenarios.json") as f:
    scenarios = json.load(f)

BASELINE_CASE_IDX = 0
baseline_case = scenarios[BASELINE_CASE_IDX]
print(f"Case : [{baseline_case['case_id']}] {baseline_case['goal'][:60]}")
print(f"Diff : {baseline_case['difficulty']} | Category: {baseline_case['category']}")
print("-" * 60)

baseline_episode = collect_episode_local(baseline_case, baseline_agent)
bs     = baseline_episode["score"]
b_algo = baseline_episode["judge_full"]["algorithmic"]

print(f"Baseline score  : {bs:.1f} / 100")
print(f"Verdict         : {baseline_episode['verdict'].upper()}")
print(f"Technique match : {b_algo.get('technique_match', 0):.1f}")
print(f"Action match    : {b_algo.get('action_match', 0):.1f}")
print(f"Completeness    : {b_algo.get('completeness', 0):.1f}")

In [ ]:
# Cell 9 — Inspect baseline response & judge feedback
resp = baseline_episode.get("response", "")
jf   = baseline_episode.get("judge_full", {})
print(f"Response length: {len(resp)} chars")
print("=" * 60)
print(resp[:1500])
if len(resp) > 1500:
    print(f"... [{len(resp)-1500} more chars]")

if jf.get("gaps"):
    print(f"\nJudge gaps:\n{jf['gaps']}")
if jf.get("strengths"):
    print(f"\nJudge strengths:\n{jf['strengths']}")


In [ ]:
# Cell 10 — RFT Training Loop (no Groq — local briefing + SBERT judge)
import time, random
from qwen_training.rft_trainer import build_lora_model, sft_step

print(f"Starting RFT: {config.rounds} rounds x {config.episodes_per_round} episodes")
print(f"Estimated time: ~{config.rounds * 15} min on T4")

# Build LoRA model once and inject into agent to avoid reloading each round
model, tokenizer = build_lora_model(config)
baseline_agent._model     = model
baseline_agent._tokenizer = tokenizer

history      = []
adapter_path = None
t0           = time.time()

for rnd in range(1, config.rounds + 1):
    print(f"\n{'='*60}\n  RFT Round {rnd}/{config.rounds}\n{'='*60}")

    round_cases = random.sample(scenarios, min(config.episodes_per_round, len(scenarios)))
    episodes    = []

    for case in round_cases:
        try:
            ep = collect_episode_local(case, baseline_agent)
            episodes.append(ep)
            print(f"  {case['case_id']} | score={ep['score']:.1f} | {ep['verdict'].upper()}")
        except Exception as e:
            print(f"  [WARN] {case['case_id']} failed: {e}")

    if not episodes:
        continue

    episodes.sort(key=lambda e: e["score"], reverse=True)
    k        = max(1, int(len(episodes) * config.top_k_ratio))
    accepted = [e for e in episodes[:k] if e["score"] >= config.min_score_threshold]

    avg_score = sum(e["score"] for e in episodes) / len(episodes)
    print(f"\n  Round {rnd} | avg={avg_score:.1f} | accepted={len(accepted)}/{len(episodes)}")

    history.append({
        "round": rnd, "episodes": len(episodes), "accepted": len(accepted),
        "avg_score": round(avg_score, 2),
        "max_score": round(max(e["score"] for e in episodes), 2),
    })

    if accepted:
        adapter_path = sft_step(model, tokenizer, accepted, config, rnd)
    else:
        print("  No episodes above threshold — skipping SFT.")

elapsed          = time.time() - t0
training_history = history
final_adapter    = adapter_path

print(f"\nRFT complete in {elapsed/60:.1f} min")
scores = [r["avg_score"] for r in training_history]
print(f"Score trajectory: {' -> '.join(f'{s:.1f}' for s in scores)}")
if len(scores) > 1:
    print(f"Net improvement : {scores[-1] - scores[0]:+.1f} points")
print(f"Final adapter   : {final_adapter}")

In [ ]:
# Cell 11 — Training summary table
# Each round dict: {round, episodes, accepted, avg_score, max_score}
import pandas as pd
from IPython.display import display

rows = []
for r in training_history:
    rows.append({
        "Round"          : r.get("round", "?"),
        "Episodes"       : r.get("episodes", "?"),
        "Accepted (SFT)" : r.get("accepted", "?"),
        "Avg score"      : f"{r.get('avg_score', 0):.1f}",
        "Max score"      : f"{r.get('max_score', 0):.1f}",
        "Accept rate"    : f"{r.get('accepted', 0) / max(r.get('episodes', 1), 1) * 100:.0f}%",
    })

df = pd.DataFrame(rows)
display(df.to_string(index=False))


In [ ]:
# Cell 12 — Plot score trajectory & acceptance rate
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("dark_background")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#070d14")

CYAN = "#00d4ff"; GREEN = "#00ff88"; RED = "#ff4444"; ORANGE = "#ff8800"; DIM = "#4a5568"

rounds     = [r.get("round", i+1)    for i, r in enumerate(training_history)]
avg_scores = [r.get("avg_score", 0)  for r in training_history]
max_scores = [r.get("max_score", 0)  for r in training_history]
accept_pct = [
    r.get("accepted", 0) / max(r.get("episodes", 1), 1) * 100
    for r in training_history
]

ax1.set_facecolor("#0d1524")
ax1.plot(rounds, avg_scores, "o-",  color=CYAN,  lw=2, ms=8, label="Avg score")
ax1.plot(rounds, max_scores, "s--", color=GREEN, lw=2, ms=8, label="Max score")
ax1.axhline(75, color=GREEN,  alpha=0.3, ls=":", label="Pass (75)")
ax1.axhline(40, color=ORANGE, alpha=0.3, ls=":", label="Min threshold (40)")
ax1.set(xlabel="Round", ylabel="Score", title="Score per Round", ylim=(0, 110))
ax1.legend(fontsize=8)

ax2.set_facecolor("#0d1524")
bar_colors = [GREEN if p >= 50 else ORANGE if p >= 25 else RED for p in accept_pct]
ax2.bar(rounds, accept_pct, color=bar_colors, alpha=0.8, edgecolor="#1a2744")
for i, (rnd, pct) in enumerate(zip(rounds, accept_pct)):
    ax2.text(rnd, pct + 1, f"{pct:.0f}%", ha="center", fontsize=9, color="white")
ax2.set(xlabel="Round", ylabel="Acceptance Rate (%)", title="Episodes Accepted for SFT", ylim=(0, 110))

for ax in [ax1, ax2]:
    ax.tick_params(colors=DIM)
    for s in ax.spines.values():
        s.set_color("#1a2744")

plt.tight_layout()
plt.savefig("training_trajectory.png", dpi=150, bbox_inches="tight", facecolor="#070d14")
plt.show()
print("Saved: training_trajectory.png")


In [ ]:
# Cell 13 — Load trained adapter & evaluate on same baseline case
from pathlib import Path
from qwen_training.qwen_target_agent import QwenTargetAgent

ckpt_dir = Path(config.output_dir)
adapters  = sorted(ckpt_dir.glob("round_*/adapter"),
                   key=lambda p: int(p.parent.name.split("_")[1]))

if adapters:
    best_adapter  = final_adapter or str(adapters[-1])
    trained_agent = QwenTargetAgent(
        model_name=config.model_name, adapter_path=best_adapter, load_in_4bit=True,
    )
    trained_agent._load()
    print(f"Loaded adapter: {best_adapter}")
else:
    trained_agent = baseline_agent
    print("No adapter found — using base model for post-eval")

post_episode = collect_episode_local(baseline_case, trained_agent)
ps = post_episode["score"]

print(f"\nSame case comparison:")
print(f"  Before training : {bs:.1f}")
print(f"  After training  : {ps:.1f}  (delta {ps-bs:+.1f})")
print(f"  Verdict         : {post_episode['verdict'].upper()}")

In [ ]:
# Cell 14 — Before / after side-by-side
print("BEFORE TRAINING:")
print("=" * 60)
print(baseline_episode.get("response", "")[:800])
print(f"Score: {bs:.1f} | {baseline_episode['verdict'].upper()}")

print("\nAFTER TRAINING:")
print("=" * 60)
print(post_episode.get("response", "")[:800])
print(f"Score: {ps:.1f} | {post_episode['verdict'].upper()}")

dim_keys = ["technique_match", "action_match", "completeness"]
print(f"\n{'Dimension':<22} {'Before':>8} {'After':>8} {'Delta':>8}")
print("-" * 50)
for d in dim_keys:
    bv = baseline_episode.get("judge_full", {}).get("algorithmic", {}).get(d, 0)
    av = post_episode.get("judge_full", {}).get("algorithmic", {}).get(d, 0)
    print(f"{d:<22} {bv:>8.1f} {av:>8.1f} {av-bv:>+8.1f}")
print(f"{'OVERALL':<22} {bs:>8.1f} {ps:>8.1f} {ps-bs:>+8.1f}")


In [ ]:
# Cell 15 — Score breakdown bar chart (before vs after)
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("#070d14")
ax.set_facecolor("#0d1524")

labels = ["Overall", "Algorithmic", "Qualitative", "Technique", "Action Match", "Completeness"]

def _get_scores(ep):
    jf   = ep.get("judge_full", {})
    algo = jf.get("algorithmic", {})
    qual = jf.get("qualitative", {})
    return [
        ep.get("score", 0),
        algo.get("total", 0),
        qual.get("total", 0),
        algo.get("technique_match", 0),
        algo.get("action_match", 0),
        algo.get("completeness", 0),
    ]

b_vals = _get_scores(baseline_episode)
a_vals = _get_scores(post_episode)

x, w = np.arange(len(labels)), 0.35
ax.bar(x - w/2, b_vals, w, label="Before RFT", color="#1a4a6b", edgecolor="#00d4ff", alpha=0.85)
ax.bar(x + w/2, a_vals, w, label="After RFT",  color="#006644", edgecolor="#00ff88", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, color="#4a5568")
ax.set(ylabel="Score", title="Score Breakdown: Before vs After RFT", ylim=(0, 110))
ax.legend()
ax.tick_params(colors="#4a5568")
for s in ax.spines.values():
    s.set_color("#1a2744")

plt.tight_layout()
plt.savefig("score_breakdown.png", dpi=150, bbox_inches="tight", facecolor="#070d14")
plt.show()
print("Saved: score_breakdown.png")


In [ ]:
# Cell 16 — Batch evaluate all 10 scenarios with trained model
import json

print(f"Evaluating {len(scenarios)} scenarios (~10-15 min on T4)...")
batch_results = []

for i, case in enumerate(scenarios):
    print(f"[{i+1:02d}/{len(scenarios)}] {case['case_id']} | {case['difficulty']:<8} | ", end="", flush=True)
    ep = collect_episode_local(case, trained_agent)
    batch_results.append(ep)
    print(f"score={ep['score']:.1f} | {ep['verdict'].upper()}")

scores   = [r["score"] for r in batch_results]
pass_cnt = sum(1 for r in batch_results if r["verdict"] == "pass")
part_cnt = sum(1 for r in batch_results if r["verdict"] == "partial")
fail_cnt = sum(1 for r in batch_results if r["verdict"] == "fail")

print(f"\nAvg score : {sum(scores)/len(scores):.1f}")
print(f"Pass / Partial / Fail: {pass_cnt} / {part_cnt} / {fail_cnt}")
print(f"Pass rate : {pass_cnt/len(scores)*100:.0f}%")

with open("batch_eval_results.json", "w") as f:
    json.dump(batch_results, f, indent=2, default=str)
print("\nSaved: batch_eval_results.json")

In [ ]:
# Cell 17 — Free-form inference demo with custom incident
GOAL = (
    "You are a SOC analyst. Investigate and respond to this incident:\n"
    "Multiple failed SSH login attempts (500+) from IP 45.33.32.156 over 30 minutes "
    "targeting the production bastion host. A successful login occurred at 03:47 UTC."
)

BRIEFING = (
    "Log Analyst  : 523 failed auth attempts, 1 success at 03:47 UTC from 45.33.32.156.\n"
    "Threat Intel : IP 45.33.32.156 flagged on AbuseIPDB (known mass-scanner).\n"
    "Vuln Scanner : Bastion runs OpenSSH 7.4 (CVE-2023-38408 agent forwarding RCE).\n"
    "Orchestrator : High-confidence brute force followed by successful credential stuffing.\n"
    "MITRE TTPs   : T1110 (Brute Force), T1021.004 (SSH), T1078 (Valid Accounts)."
)

result = trained_agent.run({"goal": GOAL, "briefing": BRIEFING})

print("QWEN RESPONSE:")
print("=" * 60)
print(result.get("response", str(result)))


In [ ]:
# Cell 18 — Score the custom response through local SBERT judge
custom_case = {
    "known_truth": {
        "techniques": [
            {"id": "T1110", "name": "Brute Force"},
            {"id": "T1021.004", "name": "SSH"},
            {"id": "T1078", "name": "Valid Accounts"},
        ],
        "correct_immediate_actions": [
            "block IP 45.33.32.156",
            "rotate SSH keys",
            "audit active sessions",
            "patch CVE-2023-38408",
            "enable MFA on bastion",
        ],
        "root_cause": "Weak password + unpatched OpenSSH agent forwarding",
        "blast_radius": "Bastion host compromised — lateral movement risk",
        "iocs": {"ips": ["45.33.32.156"], "domains": []},
    }
}

judge_result = local_judge(custom_case, result.get("response", ""))
algo = judge_result.get("algorithmic", {})

print(f"Overall score  : {judge_result.get('overall', 0):.1f} / 100")
print(f"Verdict        : {judge_result.get('verdict', '').upper()}")
print(f"Technique match: {algo.get('technique_match', 0):.1f}")
print(f"IOC match      : {algo.get('ioc_match', 0):.1f}")
print(f"Action match   : {algo.get('action_match', 0):.1f}")
print(f"Root cause     : {algo.get('root_cause_match', 0):.1f}")
print(f"Completeness   : {algo.get('completeness', 0):.1f}")

In [ ]:
# Cell 20 — Push adapter to HuggingFace Hub
import os
from pathlib import Path

HF_TOKEN = os.environ.get("HF_TOKEN", "")
HUB_REPO = config.hub_repo

if not HF_TOKEN:
    print("HF_TOKEN not set — skipping push_to_hub")
    print("Add HF_TOKEN to Colab secrets and re-run to publish")
elif not final_adapter or not Path(final_adapter).exists():
    print("No trained adapter found — run Cell 10 first")
else:
    from huggingface_hub import HfApi
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HUB_REPO, exist_ok=True, token=HF_TOKEN)
    print(f"Repo: https://huggingface.co/{HUB_REPO}")

    print("Loading model for push (may take a few min)...")
    base  = AutoModelForCausalLM.from_pretrained(config.model_name, load_in_4bit=True, device_map="auto")
    peft  = PeftModel.from_pretrained(base, final_adapter)
    tok   = AutoTokenizer.from_pretrained(config.model_name)

    peft.push_to_hub(HUB_REPO, token=HF_TOKEN)
    tok.push_to_hub(HUB_REPO, token=HF_TOKEN)

    for fname in ["training_trajectory.png", "score_breakdown.png",
                  "batch_eval_results.json", "qwen_training/checkpoints/training_history.json"]:
        if Path(fname).exists():
            api.upload_file(path_or_fileobj=fname, path_in_repo=Path(fname).name,
                            repo_id=HUB_REPO, token=HF_TOKEN)
            print(f"Uploaded {Path(fname).name}")

    print(f"Done: https://huggingface.co/{HUB_REPO}")


## Quick Reference

| Task | Cells |
|------|-------|
| Install deps | 1 |
| Clone repo | 2 |
| Set HF token (optional) | 3 |
| Train SBERT judge | 4 |
| GPU check | 5 |
| Configure RFT | 6 |
| Load baseline model | 7 |
| Baseline episode | 8-9 |
| **Full RFT training** | **10** |
| Training summary + plots | 11-12 |
| Post-training eval | 13-15 |
| Batch all 10 scenarios | 16 |
| Free-form inference | 17-18 |
| Push to HuggingFace Hub | 19 |

### Score interpretation
| Score | Verdict | Meaning |
|-------|---------|---------|
| >= 75 | PASS    | Strong incident response |
| 45-74 | PARTIAL | Adequate but gaps remain |
| < 45  | FAIL    | Insufficient response |